# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Utsabsinha19/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The ranked queue prioritizes pages for human review. Higher priority is given to pages with strong exposure, weak search position, or recent impression decline. Each recommendation includes a reason code so the content team can understand why the page was selected.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Calculate recent impression change
df["impression_change_pct"] = np.where(
    df["impressions_prev_30d"] > 0,
    (df["impressions_last_30d"] - df["impressions_prev_30d"])
    / df["impressions_prev_30d"],
    0
)

# Transparent baseline priority score
df["exposure_score"] = df["impressions_90d"].rank(pct=True)
df["position_score"] = df["avg_position"].rank(pct=True)
df["decline_score"] = (-df["impression_change_pct"]).rank(pct=True)

df["action_score"] = (
    0.40 * df["exposure_score"]
    + 0.35 * df["position_score"]
    + 0.25 * df["decline_score"]
)

def make_reason(row):
    reasons = []

    if row["exposure_score"] >= 0.75:
        reasons.append("HIGH_IMPRESSIONS")

    if row["position_score"] >= 0.75:
        reasons.append("WEAK_POSITION")

    if row["impression_change_pct"] < 0:
        reasons.append("RECENT_DECLINE")

    if not reasons:
        reasons.append("MIXED_SIGNAL")

    return "|".join(reasons)

df["reason_code"] = df.apply(make_reason, axis=1)

df = df.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = df.index + 1

queue = df[
    [
        "rank",
        "content_id",
        "action_score",
        "reason_code",
        "impressions_90d",
        "avg_position",
        "impressions_last_30d",
        "impressions_prev_30d",
        "impression_change_pct"
    ]
].copy()

queue["recommended_action"] = "REVIEW_CONTENT"

print("Ranked pages:", len(queue))
display(queue.head(20))

Ranked pages: 30000


,rank,content_id,action_score,reason_code,impressions_90d,avg_position,impressions_last_30d,impressions_prev_30d,impression_change_pct,recommended_action
0,1,content_fb66dd8f4629,0.955048,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,32518,56.5,3725,25453,-0.853652,REVIEW_CONTENT
1,2,content_fb4bf6555c79,0.951901,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,84093,45.6,4300,25249,-0.829696,REVIEW_CONTENT
2,3,content_150f89b1d73b,0.949842,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,83490,45.0,5797,32027,-0.818996,REVIEW_CONTENT
3,4,content_b80d73524f2d,0.939144,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,35620,44.0,2380,12366,-0.807537,REVIEW_CONTENT
4,5,content_a7c2dfc8a6ec,0.935393,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,76868,39.9,7653,33908,-0.774301,REVIEW_CONTENT
5,6,content_b51e2e4d22ff,0.934647,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,91795,41.2,11283,44399,-0.745873,REVIEW_CONTENT
6,7,content_095661034f9b,0.934092,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,23513,39.4,952,11326,-0.915946,REVIEW_CONTENT
7,8,content_21b3d827d451,0.931249,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,38894,39.4,3427,16654,-0.794224,REVIEW_CONTENT
8,9,content_4a50087c06cb,0.930161,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,40971,47.1,4280,14283,-0.700343,REVIEW_CONTENT
9,10,content_370de6e8e035,0.929928,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,114389,39.7,12504,45323,-0.724114,REVIEW_CONTENT


## 2. Intended use and limits

The playbook is intended for editors, SEO specialists, and content teams who need to prioritize limited review time. It can be used to create a ranked review queue and explain the main observed signals behind each recommendation. It is not valid as an automatic content-change system, a causal analysis, or a prediction of Google's ranking decisions. Recommendations should be reconsidered when the underlying data, measurement windows, or content strategy changes.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Intended users: editors, SEO specialists, content teams")
print("Purpose: prioritize pages for human review")
print("Use: decision-support")
print("Not intended for: automatic content changes or causal claims")

Intended users: editors, SEO specialists, content teams
Purpose: prioritize pages for human review
Use: decision-support
Not intended for: automatic content changes or causal claims


## 3. Human review + the no-go list

Before acting on a recommendation, a reviewer should check the page's search intent, content quality, freshness, business importance, and whether the observed performance change is meaningful enough to justify work. The reviewer should also consider whether the page has recently been updated or whether an external event could explain the change.

No-go list: never automatically rewrite, delete, redirect, or publish content based only on the model score. Never treat the score as proof of a ranking penalty, causal effect, or guaranteed traffic opportunity. Client-identifying information and private queries should not be exposed in the exported queue.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
no_go_actions = [
    "Automatic content rewriting",
    "Automatic deletion",
    "Automatic redirects",
    "Automatic publishing",
    "Claiming a causal ranking effect",
    "Treating the score as guaranteed traffic improvement"
]

print("Human review is required before action.")
print("\nNo-go list:")

for item in no_go_actions:
    print("-", item)

Human review is required before action.

No-go list:
- Automatic content rewriting
- Automatic deletion
- Automatic redirects
- Automatic publishing
- Claiming a causal ranking effect
- Treating the score as guaranteed traffic improvement


## 4. Monitoring / retrain triggers

The recommendations may become stale when the data distribution changes, the share of declining pages changes substantially, or the ranking metric falls on a consistent validation sample. A review should also be triggered when important feature distributions shift or when the content team's strategy changes. Retraining should be considered after a meaningful change in data, labels, measurement windows, or business requirements rather than on a fixed schedule alone.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Monitoring triggers:")
print("- Precision@50 declines consistently")
print("- Feature distributions change substantially")
print("- Rate of recent-decline pages changes substantially")
print("- Measurement windows or data definitions change")
print("- Content strategy or business objectives change")
print("- Human reviewers report repeated poor recommendations")

print("\nRetrain/review decision should be based on evidence of staleness.")

Monitoring triggers:
- Precision@50 declines consistently
- Feature distributions change substantially
- Rate of recent-decline pages changes substantially
- Measurement windows or data definitions change
- Content strategy or business objectives change
- Human reviewers report repeated poor recommendations

Retrain/review decision should be based on evidence of staleness.


## 5. Exports for the paper

I will export the ranked action queue so that it can be reused in the final paper and capstone documentation. The export contains the ranking, action, score, reason code, and supporting observed signals. It is intended as a reproducible decision-support artifact.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/action_playbook_queue.csv"

queue.to_csv(
    output_path,
    index=False
)

print("Export created:")
print(output_path)

print("\nRows exported:", len(queue))

display(queue.head(20))

Export created:
work/outputs/action_playbook_queue.csv

Rows exported: 30000


,rank,content_id,action_score,reason_code,impressions_90d,avg_position,impressions_last_30d,impressions_prev_30d,impression_change_pct,recommended_action
0,1,content_fb66dd8f4629,0.955048,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,32518,56.5,3725,25453,-0.853652,REVIEW_CONTENT
1,2,content_fb4bf6555c79,0.951901,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,84093,45.6,4300,25249,-0.829696,REVIEW_CONTENT
2,3,content_150f89b1d73b,0.949842,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,83490,45.0,5797,32027,-0.818996,REVIEW_CONTENT
3,4,content_b80d73524f2d,0.939144,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,35620,44.0,2380,12366,-0.807537,REVIEW_CONTENT
4,5,content_a7c2dfc8a6ec,0.935393,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,76868,39.9,7653,33908,-0.774301,REVIEW_CONTENT
5,6,content_b51e2e4d22ff,0.934647,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,91795,41.2,11283,44399,-0.745873,REVIEW_CONTENT
6,7,content_095661034f9b,0.934092,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,23513,39.4,952,11326,-0.915946,REVIEW_CONTENT
7,8,content_21b3d827d451,0.931249,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,38894,39.4,3427,16654,-0.794224,REVIEW_CONTENT
8,9,content_4a50087c06cb,0.930161,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,40971,47.1,4280,14283,-0.700343,REVIEW_CONTENT
9,10,content_370de6e8e035,0.929928,HIGH_IMPRESSIONS|WEAK_POSITION|RECENT_DECLINE,114389,39.7,12504,45323,-0.724114,REVIEW_CONTENT


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.